# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json

In [2]:
# Import modules
%reload_ext autoreload
%autoreload 2

# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from metadata import create_json
from netcdf import read_mat_file, mat_to_xarray, save_to_netcdf

In [3]:
pwd

'/Users/yugao/UOP/ORS-processing/notebooks/processing'

## INPUT REQUIRED: 
## Depth parameters from mooring diagram and mooring log 

In [4]:
instrument_height_above_bottom =  39   
water_depth_from_mooring_diagram = 4535
water_depth_from_ship_uncorrected = 4510     # uncorrected water depth 
water_depth_from_ship_corrected = 4510       # Replace with actual value, best water depth

#  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
# + 6 terminations at 0.25 ea (1.5) 
# + distance from termination on SBE cage to sensor (0.5) = 39 m

## read mat file

In [9]:
# mat_filepath = '../../data/external/stratus14/s14_sbe37_10600.mat'
mat_filepath = '../../data/external/stratus14/s14_sbe37_10601.mat'
mat_data = read_mat_file(mat_filepath)
mat_data
# Inspect the structure of mat_data
# print(mat_data.keys()) 

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Tue Oct 11 14:08:02 2016',
 '__version__': '1.0',
 '__globals__': [],
 'latitude': -19.818691666666666,
 'longitude': -84.73706666666666,
 'meta': array(('Stratus', '14', 'Robert Weller', array(('surface mooring', 'Stratus Ocean Reference Station', 14, 2015, 4510, 3.7, 60, 'AGS61 Cabo de Hornos', 'AGS61 Cabo de Hornos', '21-Apr-2015 19:58:00', '22-Jun-2016 12:06:00', array([736075.83194444, 736503.50416667]), 427.6722222222015),
       dtype=[('type', 'O'), ('experiment', 'O'), ('deployment_number', 'O'), ('start_year', 'O'), ('water_depth_m', 'O'), ('watch_circle_nm', 'O'), ('deck_height_cm', 'O'), ('deploymentcruise', 'O'), ('recoverycruise', 'O'), ('anchor_over_time', 'O'), ('anchor_release_time', 'O'), ('anchor_times', 'O'), ('duration', 'O')]), array(('WHOI', 'WHOI-UOP', 'Mooring observation', 'OceanSITES', 'Station', '38400'),
       dtype=[('institution', 'O'), ('data_assembly_center', 'O'), ('source', 'O'), ('n

## convert mat file to netcdf

In [6]:
sbe16_metadata = mat_data['meta']

In [7]:
# Use the loaded MATLAB data (mat_data_updated) for conversion
ds = mat_to_xarray(mat_data)

In [8]:
ds

<xarray.Dataset> Size: 4MB
Dimensions:  (time: 125990)
Coordinates:
  * time     (time) float64 1MB 7.361e+05 7.361e+05 ... 7.365e+05 7.365e+05
Data variables:
    temp     (time) float64 1MB 17.52 17.54 17.57 17.59 ... 21.09 21.11 21.19
    cond     (time) float64 1MB 7.4e-05 7.1e-05 7.1e-05 ... 7.1e-05 6.8e-05
    sal      (time) float64 1MB 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    depth:      4496
    latitude:   -19.818691666666666
    longitude:  -84.73706666666666

## Meta data: we will add depth paramter for deep T/S

In [7]:
depth_parameters = {
    # consult mooring diagram
    'water_depth_from_mooring_diagram': water_depth_from_mooring_diagram,  
     # uncorrected water depth
    'water_depth_from_ship_uncorrected': water_depth_from_ship_uncorrected,
    # Replace with actual value, best water depth
    'water_depth_from_ship_corrected': water_depth_from_ship_corrected,        
    # instrument_depth_from_mooring_diagram = diagram depth  - height above bottom 
    'water_depth_from_pressure' : ds.attrs['depth'],
    'instrument_depth_from_mooring_diagram': water_depth_from_mooring_diagram - instrument_height_above_bottom,  
    # instrument_depth_from_mooring_log = corrected depth (4538.97 m) - height above bottom (39 m) 
    'instrument_depth_from_mooring_log': water_depth_from_ship_corrected - instrument_height_above_bottom,  
    'instrument_height_above_bottom': 39
}

del ds.attrs['depth'] # delete depth 
depth_parameters 

{'water_depth_from_mooring_diagram': 4535,
 'water_depth_from_ship_uncorrected': 4510,
 'water_depth_from_ship_corrected': 4510,
 'water_depth_from_pressure': 4496,
 'instrument_depth_from_mooring_diagram': 4496,
 'instrument_depth_from_mooring_log': 4471,
 'instrument_height_above_bottom': 39}

In [8]:
metadata_dict = create_json(mat_data, depth_parameters)

json_path = '/Users/yugao/UOP/ORS-processing/data/metadata/stratus_OS_NTAS_2016_D_TS.json'

# Convert the dictionary to a JSON string and save it
with open(json_path, 'w') as f:
    json.dump(metadata_dict, f)

### read json file

In [9]:
with open(json_path, 'r') as f:
    metadata_from_json = json.load(f)

### incorporate the json file with all attributes

In [10]:
metadata_from_json

{'attributes': {'site': 'Stratus',
  'deployment': '14',
  'principal_investigator': 'Robert Weller',
  'institution': 'WHOI',
  'project': 'WHOI-UOP',
  'platform_type': 'surface mooring',
  'platform_year': 'Stratus Ocean Reference Station',
  'time_coverage_start': '2015-04-13T12:00:01Z',
  'time_coverage_end': '2016-06-23T23:05:00Z',
  'time_coverage_duration': 'P437D',
  'latitude_anchor_survey': -19.818691666666666,
  'longitude_anchor_survey': -84.73706666666666,
  'geospatial_lat_min': -19.818691666666666,
  'geospatial_lat_max': -19.818691666666666,
  'geospatial_lon_min': -84.73706666666666,
  'geospatial_lon_max': -84.73706666666666,
  'geospatial_vertical_min': 4496,
  'geospatial_vertical_max': 4496,
  'time_coverage_resolution': 'PT1H'},
 'depth parameters': {'water_depth_from_mooring_diagram': 4535,
  'water_depth_from_ship_uncorrected': 4510,
  'water_depth_from_ship_corrected': 4510,
  'water_depth_from_pressure': 4496,
  'instrument_depth_from_mooring_diagram': 4496,


In [11]:
# Flatten the nested dictionaries within the 'attributes' attribute
attributes_flat = {}
for key, value in  metadata_from_json.items():
    if isinstance(value, dict):
        for sub_key, sub_value in value.items():
            attributes_flat[f'{sub_key}'] = sub_value
    else:
        attributes_flat[key] = value

# Remove the 'attributes' attribute
# if 'attributes' in ds.attrs:
#     del ds.attrs['attributes']

# Update the attributes with the flattened dictionary
ds.attrs.update(attributes_flat)

In [12]:
netcdf_path = '/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus14' + mat_filepath[-16:-4] + '.nc'
netcdf_path 

'/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus14_sbe37_10601.nc'

In [13]:
# Save the dataset to netCDF
ds.to_netcdf(netcdf_path)

In [14]:
ds.attrs

{'latitude': -19.818691666666666,
 'longitude': -84.73706666666666,
 'site': 'Stratus',
 'deployment': '14',
 'principal_investigator': 'Robert Weller',
 'institution': 'WHOI',
 'project': 'WHOI-UOP',
 'platform_type': 'surface mooring',
 'platform_year': 'Stratus Ocean Reference Station',
 'time_coverage_start': '2015-04-13T12:00:01Z',
 'time_coverage_end': '2016-06-23T23:05:00Z',
 'time_coverage_duration': 'P437D',
 'latitude_anchor_survey': -19.818691666666666,
 'longitude_anchor_survey': -84.73706666666666,
 'geospatial_lat_min': -19.818691666666666,
 'geospatial_lat_max': -19.818691666666666,
 'geospatial_lon_min': -84.73706666666666,
 'geospatial_lon_max': -84.73706666666666,
 'geospatial_vertical_min': 4496,
 'geospatial_vertical_max': 4496,
 'time_coverage_resolution': 'PT1H',
 'water_depth_from_mooring_diagram': 4535,
 'water_depth_from_ship_uncorrected': 4510,
 'water_depth_from_ship_corrected': 4510,
 'water_depth_from_pressure': 4496,
 'instrument_depth_from_mooring_diagram